In [30]:
import os

os.environ['HADOOP_HOME'] = 'C:\\hadoop'
os.environ['PATH'] += os.pathsep + 'C:\\hadoop\\bin'

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test') \
        .getOrCreate()

In [4]:
spark.version

'4.1.1'

In [5]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [14]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [6]:
df = df.repartition(4)

In [7]:
df.write.parquet('yellow_tripdata_2025-11_rep.parquet', mode='overwrite')

In [9]:
df.createOrReplaceTempView('data')

In [11]:
spark.sql("""
    SELECT COUNT(*)
    FROM data
    WHERE EXTRACT(DAY FROM tpep_pickup_datetime) = 15;
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [32]:
spark.sql("""
    SELECT CAST(DATEDIFF(minute, tpep_pickup_datetime, tpep_dropoff_datetime) AS DECIMAL(10, 2)) / 60.0 AS hours
    FROM data
    ORDER BY hours DESC
    LIMIT 5;
""").show()

+---------+
|    hours|
+---------+
|90.633333|
|76.933333|
|76.200000|
|69.283333|
|67.066667|
+---------+



In [21]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

In [22]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [23]:
df_zones.createTempView("zones_data")

In [29]:
spark.sql("""
    SELECT 
        z.LocationID,
        z.Zone,
        COUNT(*) as Total_Count
    FROM data d
    INNER JOIN zones_data z ON d.PULocationID = z.LocationID
    GROUP BY z.LocationID, z.Zone
    ORDER BY Total_Count ASC
    LIMIT 5;
""").show()

+----------+--------------------+-----------+
|LocationID|                Zone|Total_Count|
+----------+--------------------+-----------+
|         5|       Arden Heights|          1|
|        84|Eltingville/Annad...|          1|
|       105|Governor's Island...|          1|
|       187|       Port Richmond|          3|
|       111| Green-Wood Cemetery|          4|
+----------+--------------------+-----------+

